In [1]:
from google.colab import drive
drive.mount('/content/drive')

!apt-get install -y ffmpeg
!pip install -q faster-whisper

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.


In [ ]:
!ffmpeg -i "/content/drive/MyDrive/ITMO/AIST/1./audio.m4a" \
  -ss 00:00:00 -to 00:45:00 -c copy \
  "/content/drive/MyDrive/ITMO/AIST/1./first_45min.mp4"

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [6]:
!pip install -q "av<19"

In [4]:
!pip install -q faster-whisper nvidia-cublas-cu12 nvidia-cudnn-cu12

import ctypes, glob, os, site

def preload_nvidia_libs():
    for sp in site.getsitepackages():
        for pattern in ["nvidia/cublas/lib/libcublas*.so*",
                        "nvidia/cudnn/lib/libcudnn*.so*"]:
            for path in sorted(glob.glob(os.path.join(sp, pattern))):
                try:
                    ctypes.CDLL(path, mode=ctypes.RTLD_GLOBAL)
                    print("загрузилась:", path)
                except OSError as e:
                    print("не загрузилась:", path, e)

preload_nvidia_libs()

загрузилась: /usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib/libcublas.so.12
загрузилась: /usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib/libcublasLt.so.12
загрузилась: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn.so.9
загрузилась: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn_adv.so.9
загрузилась: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn_cnn.so.9
загрузилась: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn_engines_precompiled.so.9
загрузилась: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn_engines_runtime_compiled.so.9
загрузилась: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn_engines_tensor_ir.so.9
загрузилась: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn_ext.so.9
загрузилась: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn_graph.so.9
загрузилась: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/l

In [ ]:
from faster_whisper import WhisperModel

model = WhisperModel("large-v3", device="cuda", compute_type="float16")

audio_path = "/content/drive/MyDrive/ITMO/AIST/1./audio.m4a"
total_duration = 2 * 60 * 60  # 2 часа в сек

segments, info = model.transcribe(
    audio_path,
    language="ru",
    vad_filter=True
)

with open("/content/drive/MyDrive/ITMO/AIST/1./transcript_full.txt", "w", encoding="utf-8") as f:
    for s in segments:
        f.write(f"[{s.start:.2f} -> {s.end:.2f}] {s.text.strip()}\n")
        progress = (s.end / total_duration) * 100
        print(f"{progress:.1f}%  |  {s.start/60:.1f} мин → {s.end/60:.1f} мин  |  {s.text.strip()[:80]}")

print("готово")

0.1%  |  0.1 мин → 0.2 мин  |  Если что, мы вас не слышим и не видим.
0.2%  |  0.2 мин → 0.2 мин  |  Да, да, я подключил.
0.2%  |  0.2 мин → 0.2 мин  |  Здравствуйте, меня зовут Легоньков Алексей.
0.3%  |  0.3 мин → 0.3 мин  |  Тема моего исследования — это влияние синтетических данных
0.3%  |  0.3 мин → 0.4 мин  |  для дообучения Table of Foundation Models.
0.3%  |  0.4 мин → 0.4 мин  |  Как было выяснено в прошлых экспериментах,
0.4%  |  0.4 мин → 0.5 мин  |  простое использование синтетических данных для дообучения
0.5%  |  0.5 мин → 0.5 мин  |  Table of Foundation Models не даёт нам прирост по метрикам.
0.5%  |  0.5 мин → 0.6 мин  |  Поэтому была выдвинута гипотеза, что мы будем использовать
0.6%  |  0.6 мин → 0.7 мин  |  синтетику для балансировки классов редких категорий.
0.7%  |  0.7 мин → 0.8 мин  |  Цель — это оценить эффективность синтетической балансировки
0.7%  |  0.8 мин → 0.8 мин  |  обучающей выборки при дообучении Table of Foundation Models
0.7%  |  0.8 мин → 0.9 мин  |

In [ ]:
pairs = [
    ("01", "00:00:12", "00:12:58"),
    ("02", "00:13:23", "00:35:52"),
    ("03", "00:36:26", "01:12:44"),
    ("04", "01:13:11", "01:33:27"),
    ("05", "01:34:08", "01:50:22"),
]

src = "/content/drive/MyDrive/ITMO/AIST/1./audio.m4a"

for name, start, end in pairs:
    out = f"/content/drive/MyDrive/ITMO/AIST/1./pair_{name}.mp4"
    !ffmpeg -y -i "{src}" -ss {start} -to {end} -c copy "{out}"
    print(f"готово: pair_{name}.mp4")

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [5]:
from faster_whisper import WhisperModel

model = WhisperModel("large-v3", device="cuda", compute_type="float16")

path = "/content/drive/MyDrive/AIST/1./"

files = [
    "pair_01.mp4",
    "pair_02.mp4",
    "pair_03.mp4",
    "pair_04.mp4",
    "pair_05.mp4"
]

for file in files:
    print("сейчас делаю", file)

    segments, info = model.transcribe(
        path + file,
        vad_filter=True
    )

    # сохраняем текст
    out_name = file.replace(".mp4", ".txt")
    with open(path + out_name, "w", encoding="utf-8") as f:
        for s in segments:
            text = f"[{s.start:.2f} -> {s.end:.2f}] {s.text.strip()}"
            f.write(text + "\n")
            print(text)

    print("готово", out_name)
    print()

сейчас делаю pair_01.mp4
[0.47 -> 9.09] Здравствуйте, меня зовут Легоньков Алексей. Тема моего исследования – это влияние синтетических данных для дообучения Table of Foundation Models.
[10.27 -> 20.55] Как было выяснено в прошлых экспериментах, простое использование синтетических данных для дообучения Table of Foundation Models не дает нам прирост по метрикам.
[20.55 -> 29.65] Поэтому была выдвинута гипотеза, что мы будем использовать синтетику для балансировки классов редких категорий.
[31.49 -> 41.11] Цель – это оценить эффективность синтетической балансировки обучающей выборки при дообучении Table of Foundation Models на задачи классификации, регрессии,
[41.11 -> 52.39] в сравнении разных методов конфигурации без дообучения на дообучение только на исходных данных и разных методов балансировки.
[53.03 -> 59.83] Задача – это подготовить набор реальных табличных датасетов, реализовать пайплан и потом все это сравнить.
[59.83 -> 73.21] Сравнивая другие исследования, наше исследование о